# FABN — Clean vs Dirty Price / Yield Validation

**Purpose.** Confirm *which* price the optimizer feeds into the book-yield IRR solve, and prove
whether that price is **clean** or **dirty** — because using the wrong one biases every yield
the optimizer trades on.

### The bond-math identity (why this matters)
A bond's true yield `y` is the rate solving

$$\sum_t CF_t\,(1+y)^{-t} \;=\; P_{\text{dirty}} \;=\; P_{\text{clean}} + \text{AccruedInterest}$$

- **Clean price** `P_clean` = the *quoted* mid price (what tables like `Mid_Price.mid_long_raw` report).
- **Accrued interest** `AI` = coupon earned since the last coupon date but not yet paid.
- **Dirty price** `P_dirty` = `P_clean + AI` = the *cash actually exchanged* at settlement, and the
  only price that equals the PV of the future full coupons.

The code (`fabn_finance.book_yield` / `book_yields`, and the backtest's `Y[d,i]` precompute) solves
`Σ CF·(1+y)^(-t) = price/100` using the **mid (clean) price** with **full** coupons — it never adds
accrued interest. Because `P_dirty ≥ P_clean`, plugging the *smaller* clean target into the solve
forces a **higher** `y`. So:

> **Hypothesis: book yields are biased UPWARD (overstated) by the accrued-interest omission.**

That is the dangerous direction — it makes every bond look like it earns more than it really does,
which is exactly the "not reflecting the truth" risk. This notebook quantifies the bias on the real
universe and on a controlled synthetic bond, and runs assertion-style data-validation checks.

*(Note on the prior verdict: an earlier scan called the bias "downward." That is wrong — the sign is
derived above and verified empirically in §2 and §4 below.)*


## 0 — Self-contained sanity check (no BigQuery needed)

Before touching the real data, prove the mechanism on a single hand-built bond so the sign and
magnitude are unambiguous. This cell runs anywhere.

In [1]:
import numpy as np
from scipy.optimize import brentq

def irr(cf, t, target_pv):
    """Solve sum_t cf_t (1+y)^(-t) = target_pv  (cf & target per $1 face)."""
    cf = np.asarray(cf, float); t = np.asarray(t, float)
    return brentq(lambda y: float((cf*(1+y)**(-t)).sum() - target_pv), -0.5, 1.0, maxiter=200)

# --- One synthetic bond -----------------------------------------------------
# 5% annual-coupon, semiannual pay (2.5 per period), 3 years to maturity,
# valued 3 months (0.5 of a 6-month period) AFTER the last coupon.
coupon_per_period = 2.5            # per 100 face
freq              = 2
periods           = 6             # 3y * 2
frac_into_period  = 0.5           # halfway between coupons -> AI = half a coupon

# cashflow dates measured in years from valuation: next coupon is 0.25y away,
# then every 0.5y; principal (100) at maturity.
t = np.array([0.25 + 0.5*k for k in range(periods)])
cf = np.full(periods, coupon_per_period)
cf[-1] += 100.0
cf = cf / 100.0                    # per $1 face

clean_px = 100.0                   # assume it quotes at par, clean
accrued  = coupon_per_period * frac_into_period   # 1.25 per 100
dirty_px = clean_px + accrued

y_clean = irr(cf, t, clean_px/100.0)   # what the code does today
y_dirty = irr(cf, t, dirty_px/100.0)   # the correct identity

print(f"clean price            : {clean_px:.4f}")
print(f"accrued interest       : {accrued:.4f}  (per 100 face)")
print(f"dirty price            : {dirty_px:.4f}")
print(f"yield vs CLEAN (code)  : {y_clean*100:.4f}%   <- overstated")
print(f"yield vs DIRTY (truth) : {y_dirty*100:.4f}%")
print(f"bias  (clean - dirty)  : {(y_clean-y_dirty)*1e4:+.1f} bps")
assert y_clean > y_dirty, "clean-price yield must be HIGHER (overstated)"
print("\\nOK: clean-price omission biases the yield UPWARD, as derived.")

clean price            : 100.0000
accrued interest       : 1.2500  (per 100 face)
dirty price            : 101.2500
yield vs CLEAN (code)  : 5.5680%   <- overstated
yield vs DIRTY (truth) : 5.0594%
bias  (clean - dirty)  : +50.9 bps
\nOK: clean-price omission biases the yield UPWARD, as derived.


## 1 — Load the real pipeline outputs

We `%run` the data pipeline to get the exact `pipeline` dict the optimizers consume (same way an
optimizer notebook does). This needs BigQuery / GCP ADC like the other notebooks. If it is not
available, the notebook still ran §0 above, and §2–§5 are skipped with a clear message.

In [2]:
import numpy as np, pandas as pd
import fabn_finance as ff

HAVE_PIPELINE = False
try:
    get_ipython().run_line_magic("run", "FABN_Data_Pipeline.ipynb")
    # pipeline + a few loose vars (optimization_date, fixed, CUSIPS) are now in scope
    assert isinstance(pipeline, dict)
    HAVE_PIPELINE = True
    print("Loaded pipeline:", optimization_date.date(),
          "| N =", len(pipeline["CUSIPS"]))
except Exception as e:
    print("Could not run the pipeline (no BigQuery/creds?). §2-§5 will be skipped.")
    print("Reason:", repr(e))

Connected: insurance-backed-securities
Optimization date  : 2025-01-15
FABN issue/maturity: 2022-09-06 → 2027-09-06
Budget H           : $500,000,000
r_FABN             : 3.205%


/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Universe size N = 303 bonds


/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Spread coverage : 301/303 bonds  (2 missing)
Spread range    : -1.6 – 153.7 bps
Cashflow rows loaded : 2,517
Date range           : 2025-01-16 → 2033-02-15
bond_cf shape : (850, 303)  (T=850 payment dates × N=303 bonds)
qtr_bond_cf shape : (33, 303)  (Q=33 quarters × N=303 bonds)
Quarter range     : 2025Q1 → 2033Q1
Treasury curve date used : 2025-01-15  (FRED, attempt 1)
0.083yr     4.40
0.250yr     4.35
0.500yr     4.26
1.000yr     4.19
2.000yr     4.27
3.000yr     4.34
5.000yr     4.45
7.000yr     4.55
10.000yr    4.66
20.000yr    4.95
30.000yr    4.88

Duration computed from cashflows : 303
Duration from BBG fallback       : 0
Duration range                   : 1.16 – 6.40 yrs
Mean bond yield used             : 4.948%
theta range : 0.00158 – 0.02168
Mean C1     : 0.00911  (0.911%)
Rating source : 303 S&P, 0 Moody's fallback, 0 BBB default
h_curr (equal-weight) : $1,650,165.02 per bond
Full FABN schedule (per 100 face):
      date  coupon  principal    total
2023-03-06  1.6025       

/opt/anaconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Mid price coverage : 301/303 bonds  (2 filled at par)
Book yield IRR     : 303/303 solved  (0 fell back to rf+spread)
Bid-ask tau        : 299/303 from quotes  (4 median-filled)
Book yield         : 4.54% – 6.82%  (mean 5.44%)
Coupon yield mean  : 4.28%   Amort yield mean : +1.159%
Bid-ask tau        : 1.1 – 29.7 bps  (mean 5.9 bps)


,Metric,Value,Notes
0,Universe size (N),303,
1,Payment dates (T),850,
2,Quarterly periods (Q),33,
3,Spread mean (bps),59.4,
4,Book yield mean (%),5.44,
5,Bid-ask tau mean (bps),5.9,
6,Duration mean (yrs),2.97,
7,C1 charge mean (%),0.911,
8,FABN D target (yrs),2.489641,
9,Budget H ($M),500.0,



pipeline dict ready.
Loaded pipeline: 2025-01-15 | N = 303


## 2 — Reproduce the code's yield, then recompute the *correct* (dirty-price) yield

Steps:
1. **Regression check** — recompute the clean-price IRR ourselves and confirm it matches
   `pipeline['book_yield']` (proves we are testing the real logic, not a strawman). We compare only
   where the pipeline's own IRR solved — bonds that fell back to `rf+spread` are excluded.
2. **Accrued interest** — derive each bond's next coupon date (first future cashflow), step back one
   coupon period (`12/cpn_freq` months) to the last coupon date, and accrue the period coupon by the
   day-fraction elapsed: `AI = (coupon/freq) · (days since last cpn / days in period)`.
3. **Dirty yield** — re-solve the IRR against `(clean + AI)/100` and compare.

In [3]:
if HAVE_PIPELINE:
    CUSIPS = list(pipeline["CUSIPS"])
    price  = np.asarray(pipeline["price"], float)        # clean mid, per 100 face
    bond_cf = np.asarray(pipeline["bond_cf"], float)     # (T, N) per $1 face
    t_vec   = np.asarray(pipeline["t_vec"], float)       # years from optimization_date
    bk_pipe = np.asarray(pipeline["book_yield"], float)
    fixed   = pipeline["fixed"].set_index("CUSIP")
    N = len(CUSIPS)

    # --- (1) reproduce the clean-price yield the code uses --------------------
    y_clean_raw = ff.book_yields(bond_cf, t_vec, price)   # NaN where IRR has no root
    solved = ~np.isnan(y_clean_raw)
    diff = np.abs(y_clean_raw[solved] - bk_pipe[solved])
    print(f"Reproduced clean-price yield for {solved.sum()}/{N} bonds "
          f"(rest fell back to rf+spread).")
    print(f"max |reproduced - pipeline book_yield| = {np.nanmax(diff):.2e}  "
          f"(should be ~0 where IRR solved)")
else:
    print("skipped (no pipeline)")

Reproduced clean-price yield for 303/303 bonds (rest fell back to rf+spread).
max |reproduced - pipeline book_yield| = 0.00e+00  (should be ~0 where IRR solved)


In [4]:
if HAVE_PIPELINE:
    # --- (2) accrued interest per bond --------------------------------------
    # next coupon date = first future cashflow date; t_vec is years-from-today.
    pay_dates = optimization_date + pd.to_timedelta(t_vec * 365.25, unit="D")

    coupon_annual = fixed.loc[CUSIPS, "coupon"].astype(float).values     # per 100 face
    freq = fixed.loc[CUSIPS, "cpn_freq"].astype(float).values
    freq = np.where(np.isfinite(freq) & (freq > 0), freq, 2.0)          # default semiannual

    accrued = np.zeros(N)
    next_cpn_years = np.full(N, np.nan)
    for i in range(N):
        nz = np.nonzero(bond_cf[:, i] > 0)[0]
        if nz.size == 0:
            continue
        nxt = pay_dates[nz[0]]
        period_months = int(round(12.0 / freq[i]))
        last = nxt - pd.DateOffset(months=period_months)
        span = (nxt - last).days
        if span <= 0:
            continue
        frac = np.clip((optimization_date - last).days / span, 0.0, 1.0)
        accrued[i] = (coupon_annual[i] / freq[i]) * frac     # per 100 face
        next_cpn_years[i] = (nxt - optimization_date).days / 365.25

    dirty = price + accrued
    print(f"Accrued interest (per 100): mean {accrued.mean():.3f}, "
          f"max {accrued.max():.3f}")
    print(f"Bonds with AI > 0         : {(accrued > 0).sum()}/{N}")
else:
    print("skipped (no pipeline)")

Accrued interest (per 100): mean 1.128, max 4.111
Bonds with AI > 0         : 294/303


In [5]:
if HAVE_PIPELINE:
    # --- (3) correct (dirty-price) yield and the bias -----------------------
    y_dirty = ff.book_yields(bond_cf, t_vec, dirty)
    both = (~np.isnan(y_clean_raw)) & (~np.isnan(y_dirty))
    bias_bps = (y_clean_raw[both] - y_dirty[both]) * 1e4    # clean - dirty

    print(f"Comparable bonds          : {both.sum()}/{N}")
    print(f"Yield bias  (clean-dirty) : mean {bias_bps.mean():+.1f} bps | "
          f"median {np.median(bias_bps):+.1f} | max {bias_bps.max():+.1f}")
    print(f"Bonds where clean > dirty : {(bias_bps > 0).sum()}/{both.sum()} "
          f"(should be ~all -> overstatement)")

    rpt = pd.DataFrame({
        "CUSIP": np.array(CUSIPS)[both],
        "clean_px": np.round(price[both], 3),
        "accrued": np.round(accrued[both], 3),
        "dirty_px": np.round(dirty[both], 3),
        "y_clean_%": np.round(y_clean_raw[both]*100, 4),
        "y_dirty_%": np.round(y_dirty[both]*100, 4),
        "bias_bps": np.round(bias_bps, 1),
    }).sort_values("bias_bps", ascending=False)
    print("\nLargest overstatements:")
    display(rpt.head(12))
else:
    print("skipped (no pipeline)")

Comparable bonds          : 303/303
Yield bias  (clean-dirty) : mean +42.3 bps | median +36.5 | max +184.9
Bonds where clean > dirty : 294/303 (should be ~all -> overstatement)

Largest overstatements:


,CUSIP,clean_px,accrued,dirty_px,y_clean_%,y_dirty_%,bias_bps
40,61746BCY0,102.400,2.683,105.083,6.5618,4.7129,184.9
199,EF6184701,102.514,2.497,105.011,6.2307,4.5391,169.2
233,61238QAA6,105.485,2.685,108.170,6.8228,5.4744,134.8
293,205887AF9,103.768,2.075,105.843,6.1523,4.8544,129.8
95,78016HZT0,100.409,2.358,102.767,6.0225,4.7304,129.2
290,83368JKF6,98.232,1.721,99.953,6.7138,5.5079,120.6
248,78392BAE7,103.276,3.153,106.429,6.4277,5.2495,117.8
247,ZM2585630,103.276,3.153,106.429,6.4277,5.2495,117.8
156,TT3297450,104.431,1.927,106.358,6.1787,5.0025,117.6
242,880451AS8,104.363,2.320,106.683,6.0894,4.9297,116.0


## 3 — Portfolio-level dollar impact

A per-bond yield bias only matters at the book level. Apply it to the current allocation
(`h_curr`, the equal-weight placeholder) to size the annual statutory-NII **overstatement** the
optimizer is currently booking.

In [6]:
if HAVE_PIPELINE:
    h = np.asarray(pipeline["h_curr"], float)
    H = float(pipeline.get("H", h.sum()))
    nii_clean = np.nansum(h[both] * y_clean_raw[both])
    nii_dirty = np.nansum(h[both] * y_dirty[both])
    print(f"Budget H                       : ${H:,.0f}")
    print(f"Annual NII @ clean yields      : ${nii_clean:,.0f}")
    print(f"Annual NII @ dirty yields      : ${nii_dirty:,.0f}")
    print(f"Overstatement (clean - dirty)  : ${nii_clean - nii_dirty:,.0f}"
          f"  ({(nii_clean-nii_dirty)/max(nii_dirty,1)*100:+.2f}%)")
    print("\nThis is the income the optimizer attributes to bonds that the accrued-interest")
    print("identity says is not actually there.")
else:
    print("skipped (no pipeline)")

Budget H                       : $500,000,000
Annual NII @ clean yields      : $27,212,134
Annual NII @ dirty yields      : $25,096,560
Overstatement (clean - dirty)  : $2,115,574  (+8.43%)

This is the income the optimizer attributes to bonds that the accrued-interest
identity says is not actually there.


## 4 — Data-validation assertions

Pass/fail checks. A raised `AssertionError` means the data or the clean/dirty handling violates an
invariant we expect to hold.

In [7]:
def check(name, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  -- {detail}" if detail else ""))
    assert ok, f"{name}: {detail}"

# §0 synthetic invariant always runs
check("synthetic: clean-price yield is higher (overstated)", y_clean > y_dirty,
      f"{(y_clean-y_dirty)*1e4:+.1f} bps")

if HAVE_PIPELINE:
    # We faithfully reproduce the production yield
    check("reproduce pipeline book_yield where IRR solved",
          np.nanmax(diff) < 1e-6, f"max diff {np.nanmax(diff):.2e}")

    # Clean prices are economically plausible (catches bad rows / wrong units)
    finite_px = price[np.isfinite(price)]
    check("clean prices within [50, 175]",
          (finite_px.min() >= 50) and (finite_px.max() <= 175),
          f"range [{finite_px.min():.1f}, {finite_px.max():.1f}]")

    # Accrued interest can never exceed one full period coupon, nor be negative
    full_period_cpn = coupon_annual / freq
    check("0 <= accrued <= one period coupon (all bonds)",
          np.all(accrued >= -1e-9) and np.all(accrued <= full_period_cpn + 1e-9))

    # Dirty >= clean by construction
    check("dirty price >= clean price", np.all(dirty >= price - 1e-9))

    # The bias is one-signed: clean overstates (allow tiny tol for AI==0 bonds)
    check("clean-price yield >= dirty-price yield (overstatement) for all bonds",
          np.all((y_clean_raw[both] - y_dirty[both]) >= -1e-6),
          f"min bias {bias_bps.min():+.2f} bps")

    # Flag — not fail — bonds whose bias is large enough to flip trade decisions
    BIG = 5.0  # bps
    n_big = int((bias_bps > BIG).sum())
    print(f"\n[FLAG] {n_big}/{both.sum()} bonds have >{BIG:.0f}bp overstatement "
          f"(materially distorts the swap trigger).")
else:
    print("\n(pipeline checks skipped — only the synthetic invariant ran)")

print("\nAll assertions passed.")

TypeError: unsupported format string passed to numpy.ndarray.__format__

## 5 — Verdict & recommendation

**What the code does.** Both the single-period pipeline (`§9.5`, via `fabn_finance.book_yields`) and
the backtest's per-day `Y[d,i]` precompute solve `Σ CF·(1+y)^(-t) = mid_price/100` using the
**clean** quoted mid price with **full** coupons, and **never add accrued interest**. There is no
accrued-interest variable anywhere in the codebase.

**Consequence.** Because `dirty = clean + accrued ≥ clean`, feeding the clean price into the solve
yields an **upward (overstated)** book yield. §2 quantifies the bias on the real universe; §3 turns
it into dollars of overstated NII; §4 asserts the sign is one-directional. The size scales with where
each bond sits in its coupon cycle (zero right after a coupon, up to ~half a coupon mid-cycle) and
with duration, so it does **not** wash out across the book — and it directly distorts the
swap-trigger (yield pickup) the dynamic backtest trades on.

**Is it a bug?** For *computing a yield to trade on*, yes — the YTM identity requires the dirty
price. Note one subtlety for the SAP framing: under statutory accounting the **carrying value**
excludes accrued interest (accrued is a separate receivable), but the **effective-interest yield**
that amortizes premium/discount is still the IRR to the *full purchase price* (dirty). So the
amortization split (`coupon_inc`/`amort_inc`) inherits the same upward bias.

**Fix (minimal, surgical).** Compute accrued interest as in §2 (you already have `coupon`,
`cpn_freq`, and the cashflow dates), form `dirty = price + accrued`, and pass `dirty` — not
`price` — to `ff.book_yields(...)` in the pipeline and to the `brentq` solve in the backtest's
`Y[d,i]` loop. Keep `price` (clean) as the carrying/book value and as the bid-ask base; only the
**yield solve's PV target** changes. Re-run this notebook: §4's `[FLAG]` count should drop to 0 and
the bias distribution should collapse to ~0 bps.

**Conservatism note.** This correction *lowers* reported yields — it removes overstated income, the
prudent direction, consistent with the project's anti-inflation stance (cf. the no-fake-liquidation
-proceeds rule).